In [2]:
import os #NAO RODA
if not os.path.exists('Faceswap-Deepfake-Pytorch'):
    !wget -q https://www.dropbox.com/s/5ji7jl7httso9ny/person_images.zip
    !wget -q https://raw.githubusercontent.com/sizhky/deep-fake-util/main/random_warp.py
    !unzip -q person_images.zip
!pip install -q torch_snippets torch_summary
from torch_snippets import *
from random_warp import get_training_data 
import cv2 #imports
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml') #carrega os dados
 
def crop_face(img): 
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) #tons de cinza na imagem
    faces = face_cascade.detectMultiScale(gray, 1.3, 5) #pega os rostos
    if(len(faces) > 0): #se pegou algum rosto
        for (x,y,w,h) in faces: 
            img2 = img[y:(y+h),x:(x+w),:] #pega so ao rosto
        img2 = cv2.resize(img2,(256,256))
        return img2, True
    else:
        return img, False

[person_images.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of person_images.zip or
        person_images.zip.zip, and cannot find person_images.zip.ZIP, period.

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [ ]:
    def crop_images(folder):
        images = Glob(folder+'/*.jpg') #pega as fotos
        for i in range(len(images)): #pega as imagens ve se achou o rosto ou nao
            img = read(images[i],1)
            img2, face_detected = crop_face(img)
            if(face_detected==False):
                continue
            else:
                cv2.imwrite('cropped_faces_'+folder+'/'+str(i)+'.jpg',cv2.cvtColor(img2, cv2.COLOR_RGB2BGR)) #salva o rosto
    crop_images('personA')
    crop_images('personB')

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, items_A, items_B):
        self.items_A = np.concatenate([read(f,1)[None] for f in items_A])/255.
        self.items_B = np.concatenate([read(f,1)[None] for f in items_B])/255.
        self.items_A += self.items_B.mean(axis=(0, 1, 2)) - self.items_A.mean(axis=(0, 1, 2))  #pega os 2 rostos e arruma a media

    def __len__(self):
        return min(len(self.items_A), len(self.items_B)) #menor conjunto de imagens
    def __getitem__(self, ix): #pega um par
        a, b = choose(self.items_A), choose(self.items_B)
        return a, b

    def collate_fn(self, batch): #pega os dados pra treinar
        imsA, imsB = list(zip(*batch))
        imsA, targetA = get_training_data(imsA, len(imsA))
        imsB, targetB = get_training_data(imsB, len(imsB))
        imsA, imsB, targetA, targetB = [torch.Tensor(i).permute(0,3,1,2).to(device) for i in [imsA, imsB, targetA, targetB]]
        return imsA, imsB, targetA, targetB

a = ImageDataset(Glob('cropped_faces_personA'), Glob('cropped_faces_personB')) #cria o dataset
x = DataLoader(a, batch_size=32, collate_fn=a.collate_fn) #cria um dataloader com o batch

In [ ]:
inspect(*next(iter(x))) #mostra info

for i in next(iter(x)):
    subplots(i[:8], nc=4, sz=(4,2)) #pega 8 imagens

In [ ]:
def _ConvLayer(input_features, output_features):
    return nn.Sequential(
        nn.Conv2d(input_features, output_features, kernel_size=5, stride=2, padding=2),
        nn.LeakyReLU(0.1, inplace=True)
    ) #camada que convolui

def _UpScale(input_features, output_features): #auemnta a resolucao da imagem
    return nn.Sequential(
        nn.ConvTranspose2d(input_features, output_features, kernel_size=2, stride=2, padding=0),
        nn.LeakyReLU(0.1, inplace=True)
    )

class Reshape(nn.Module): #remodela ela
    def forward(self, input):
        output = input.view(-1, 1024, 4, 4) 
        return output

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self):
        super(Autoencoder, self).__init__()

        self.encoder = nn.Sequential(
            _ConvLayer(3, 128),
            _ConvLayer(128, 256),
            _ConvLayer(256, 512),
            _ConvLayer(512, 1024), #convolucao
            nn.Flatten(), #passa pra vetor
            nn.Linear(1024 * 4 * 4, 1024),
            nn.Linear(1024, 1024 * 4 * 4), #camada totalmente conectada
            Reshape(),
            _UpScale(1024, 512), #aumento da imagme
        )

        self.decoder_A = nn.Sequential(
            _UpScale(512, 256),
            _UpScale(256, 128),
            _UpScale(128, 64), #aumenta a iamgem
            nn.Conv2d(64, 3, kernel_size=3, padding=1), #convolucao
            nn.Sigmoid(), #ativacao
        )

        self.decoder_B = nn.Sequential(
            _UpScale(512, 256),
            _UpScale(256, 128),
            _UpScale(128, 64), #aumenta a imagem
        nn.Sigmoid(),
    )

    def forward(self, x, select='A'): #fluxo do modelo
        if select == 'A':
            out = self.encoder(x)
            out = self.decoder_A(out)
        else:
            out = self.encoder(x)
            out = self.decoder_B(out)
        return out

# Do video so consegui pegar isso, ele tava muito cortado e esse codigo simplesmente nao funciona